In [ ]:
# Add various stellar masses and SFRs to the mock catalogue

In [ ]:
import numpy as np
from astropy.table import Table
from astropy.io import fits
import vacmatching as vacm
import matplotlib.pyplot as plt
from astropy.table import join

In [ ]:
#Define which mock catalogue we are going to add stellar masses to and which vac files to read them from
reg="S"
obs_seed= 123
sig_lgv = 0.02
min_Vpeak=70.0       #Discard all halos with Vpeak<min_Vpeak. This limits the redshift below which the mock is complete  
N_neigh_vpeak = 1000 #with 1000 the logVpeak scatter among neighbours is < 10^-6 and so will introduce no more than 10^-5 scatter in the magnitude relation
mockcat=rf'./sdata/Mocks/mock_{reg}_seed{obs_seed:3d}_sig_lgv{sig_lgv:.2f}.fits'#file to store resulting mock catalogue
mockcat_out =mockcat.replace('.fits', '_augmented.fits')

#FSF
fsfcat ='/global/cfs/cdirs/desicollab/science/gqp/y3gqp/v1.0/parent/BGS_ANY-y3gqp-parent-fsf-v1.0.fits'

#CIGALE
cigcat ='/global/cfs/cdirs/desicollab/science/gqp/y3gqp/v1.0/st_mass/parent/BGS-y3gqp-parent-v.1.0-cigale-MS-v1.0.fits'




In [ ]:
# Read the mock catalogue
mock=Table.read(mockcat)
mock.info('stats')

In [ ]:
# Read the required columns from the CIGALE VAC
cig_cols = ['TARGETID', 'LOGM', 'LOGM_ERR', 'LOGSFR', 'LOGSFR_ERR', 'CHI2']

with fits.open(cigcat, memmap=True) as hdul:
    data = hdul[1].data
    cig = Table({c: data[c] for c in cig_cols})

cig.info('stats')
#Remove rows with LOGM=0 and CHI2>17
mask=(cig['LOGM']>4) &  (cig['CHI2']<17.0)  
cig=cig[mask]

cig.info('stats')



In [ ]:
# Read required columns from the FSF VAC SPECPHOT Table

fsf_cols = ['TARGETID', 'LOGMSTAR', 'LOGMSTAR_IVAR', 'SFR', 'SFR_IVAR']

with fits.open(fsfcat, memmap=True) as hdul:
    data = hdul['SPECPHOT'].data
    fsf = Table({c: data[c] for c in fsf_cols})

#Remove rows with TARGETID=999999
mask=(fsf['TARGETID']!=999999)   
fsf=fsf[mask]
fsf.info('stats')

In [ ]:
# left join the mock table with the cig and fsf tables using TARGETID as the key
print("Initial:       ", len(mock))

mock = join(mock, cig, keys='TARGETID', join_type='left')
print("After CIG join:", len(mock))

mock = join(mock, fsf, keys='TARGETID', join_type='left')
print("After FSF join:", len(mock))


mock.info('stats')

In [ ]:
# Attempt to find donor objects to assign properties to the objects that do not have a match
# by searching for objects that are close in Z, ABSMAG and restframe colour.
#
print('For CIGALE VAC:')
# Properties to be copied from the selected donor galaxy.
fill_cols_cig = cig_cols[1:]  # we just remove the first column which is TARGETID as we do not need a copy of it
# Define a mask that flags the rows with no match and the rows with LOGM=0 
bad_cig=vacm.find_bad_rows(mock,col='LOGM')
# Find donors and copy properties
mock,donor1_cig=vacm.assign_donors_first_pass(mock,bad_cig,fill_cols_cig,dz=0.02,dmag=0.1,dcol=0.1)

print()
print('For FSF VAC:')
#Now repeat for FSF VAC
fill_cols_fsf = fsf_cols[1:]
# Define a mask that flags the rows with no match and the rows with LOGM=0 
bad_fsf=vacm.find_bad_rows(mock,col='LOGMSTAR')
# Find donors and copy properties
mock,donor1_fsf=vacm.assign_donors_first_pass(mock,bad_fsf,fill_cols_fsf,dz=0.02,dmag=0.1,dcol=0.1)

In [ ]:
# For the remaining unmatched objects make a second pass to find donors where we keep the dmag and dcol limits 
# but take the object closest in redshift within this magnitude-colour box
mock=vacm.assign_donors_second_pass(mock,bad_cig,fill_cols_cig,dmag=0.1,dcol=0.1)
mock=vacm.assign_donors_second_pass(mock,bad_fsf,fill_cols_fsf,dmag=0.1,dcol=0.1)

In [ ]:
# Define a columns of flags that indicate how the matching objects were found
mock=vacm.add_provenance_flags(mock,bad_cig,donor1_cig,flag='cigflag')
mock=vacm.add_provenance_flags(mock,bad_fsf,donor1_fsf,flag='fsfflag')

In [ ]:
# Scatter plot of the remaining problematic objects for which  no donor could be found.
# It shows they have weird absolute magnitudes probably as a result of an incorrect redshift
good=(mock['cigflag']>0)
remaining= (mock['cigflag']<0)

print("Remaining:", remaining.sum())

colour = mock['ABSMAG_GP1'] - mock['ABSMAG_RP1']

plt.figure(figsize=(6,5))
plt.scatter(
    mock['ABSMAG_RP1'][good],
    colour[good],
    s=0.1,
    alpha=0.1
)

plt.scatter(
    mock['ABSMAG_RP1'][remaining],
    colour[remaining],
    c='red',
    s=20
)

plt.xlabel('ABSMAG_RP1')
plt.ylabel('g-r')

In [ ]:
# Plot of FSF stellar mass versus CIGALE stellar mass
# with the objects assigned by the donor process and the objets for which no donor could be found
# overlaid
 


unresolved= (mock['cigflag']<0)
print("Remaining for which no donor found:", unresolved.sum())

second_pass_filled = (mock['cigflag']==3)

matched = (mock['cigflag']==1)
filled_by_donor = (mock['cigflag']==2)

plt.figure(figsize=(7, 6))

plt.hexbin(
    mock['LOGMSTAR'][matched],
    mock['LOGM'][matched],
    gridsize=150,
    bins='log',
    mincnt=1
)


# Objects that received donor values
plt.scatter(
    mock['LOGMSTAR'][filled_by_donor],
    mock['LOGM'][filled_by_donor],
    s=1,
    c='tab:red',
    alpha=0.04
)


# Dummy artist for legend
plt.scatter(
    [], [],
    s=3,
    alpha=1,
    c='red',
    label=f'1st pass Donor-filled ({filled_by_donor.sum()})'
)

# Objects that received donor values
plt.scatter(
    mock['LOGMSTAR'][second_pass_filled],
    mock['LOGM'][second_pass_filled],
    s=3,
    c='tab:cyan',
    alpha=0.4,
    label=f'2nd pass Donor-filled ({second_pass_filled.sum()})'
)


# Remaining unresolved objects
plt.scatter(
    mock['LOGMSTAR'][unresolved],
    np.full(unresolved.sum(), 5),   # arbitrary y-position
    s=30,
    c='tab:cyan',
    marker='x',
    label=f'Unresolved ({unresolved.sum()})'
)

plt.colorbar(label='N')
plt.xlabel(r'$FSF \log M_\star$')
plt.ylabel(r'$CIGALE \log M_\star$')
plt.title(rf'Matched objects ({matched.sum()})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot of absolute magnitude versus stellar mass
# with the objects assigned by the donor process and the objets for which no donor could be found
# overlaid
 


unresolved= (mock['cigflag']<0)
print("Remaining for which no donor found:", unresolved.sum())

second_pass_filled = (mock['cigflag']==3)

matched = (mock['cigflag']==1)
filled_by_donor = (mock['cigflag']==2)

plt.figure(figsize=(7, 6))

plt.hexbin(
    mock['ABSMAG_RP1'][matched],
    mock['LOGM'][matched],
    gridsize=150,
    bins='log',
    mincnt=1
)


# Objects that received donor values
plt.scatter(
    mock['ABSMAG_RP1'][filled_by_donor],
    mock['LOGM'][filled_by_donor],
    s=1,
    c='tab:red',
    alpha=0.04
)


# Dummy artist for legend
plt.scatter(
    [], [],
    s=3,
    alpha=1,
    c='red',
    label=f'1st pass Donor-filled ({filled_by_donor.sum()})'
)

# Objects that received donor values
plt.scatter(
    mock['ABSMAG_RP1'][second_pass_filled],
    mock['LOGM'][second_pass_filled],
    s=3,
    c='tab:cyan',
    alpha=0.4,
    label=f'2nd pass Donor-filled ({second_pass_filled.sum()})'
)


# Remaining unresolved objects
plt.scatter(
    mock['ABSMAG_RP1'][unresolved],
    np.full(unresolved.sum(), 5),   # arbitrary y-position
    s=30,
    c='tab:cyan',
    marker='x',
    label=f'Unresolved ({unresolved.sum()})'
)

plt.colorbar(label='N')
plt.xlabel(r'$M_r$ (ABSMAG_RP1)')
plt.ylabel(r'$\log M_\star$')
plt.title(rf'Matched objects ({matched.sum()})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
colour = mock['ABSMAG_GP1'] - mock['ABSMAG_RP1']
ssfr_proxy = mock['LOGSFR'] - mock['LOGM']

plt.figure(figsize=(7, 6))

plt.hexbin(
    colour[matched],
    ssfr_proxy[matched],
    gridsize=150,
    bins='log',
    mincnt=1
)



# Objects that received donor values
plt.scatter(
    colour[filled_by_donor],
    ssfr_proxy[filled_by_donor],
    s=1,
    c='tab:red',
    alpha=0.04
)


# Dummy artist for legend
plt.scatter(
    [], [],
    s=3,
    alpha=1,
    c='red',
    label=f'1st pass Donor-filled ({filled_by_donor.sum()})'
)

# Objects that received donor values
plt.scatter(
    colour[second_pass_filled],
    ssfr_proxy[second_pass_filled],
    s=3,
    c='tab:cyan',
    alpha=0.4,
    label=f'2nd pass Donor-filled ({second_pass_filled.sum()})'
)


# Remaining unresolved objects
plt.scatter(
    colour[unresolved],
    np.full(unresolved.sum(), -16),   # arbitrary y-position
    s=30,
    c='tab:cyan',
    marker='x',
    label=f'Unresolved ({unresolved.sum()})'
)

plt.colorbar(label='N')
plt.xlabel(r'$g-r$')
plt.ylabel(r'$\log\mathrm{SFR} - \log M_\star$')
plt.title(rf'Matched objects ({matched.sum()})')
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

cols = ['Zobs', 'rmag', 'RA', 'DEC']

nomatch1= np.isnan(mock['LOGM'])

for ax, col in zip(axes.flat, cols):

    ax.hist(
        mock[col],
        bins=50,
        density=True,
        histtype='step',
        lw=2,
        label='All'
    )

    ax.hist(
        mock[col][nomatch1],
        bins=50,
        density=True,
        histtype='step',
        lw=2,
        label='No match and no donor'
    )

    ax.set_xlabel(col)
    ax.set_ylabel('Normalized density')

axes[0,0].legend()

plt.tight_layout()
plt.show()

In [ ]:
#Save the augmented catalogue
mock.write(mockcat_out, overwrite=True)